# Group2 Baseline Workflow (Orchestration)

This notebook keeps heavy logic in `src/group2_stage2` and only orchestrates steps.


## Stage 0 Checklist
- [ ] Group1 environment (`.venv`) is active
- [ ] `configs/workflow_paths.json` is updated
- [ ] Stage2 variant datasets exist under `stage2_root/<variant>/stage2_dataset.jsonl`


In [ ]:
import json
import sys
from pathlib import Path

# Resolve project root robustly even if notebook cwd changes.
project_root = Path.cwd().resolve()
if (project_root / "configs" / "workflow_paths.json").exists():
    pass
elif (project_root.parent / "configs" / "workflow_paths.json").exists():
    project_root = project_root.parent
else:
    # Fallback for unusual launch locations.
    project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

config_path = project_root / "configs" / "workflow_paths.json"
raw_cfg = json.loads(config_path.read_text(encoding="utf-8"))

cfg = {
    k: (v.replace("${PROJECT_ROOT}", str(project_root)) if isinstance(v, str) else v)
    for k, v in raw_cfg.items()
}

stage2_root = Path(cfg["stage2_root"])
image_root = Path(cfg["image_root"])
feature_root = Path(cfg["clip_feature_root"])
quality_variants = cfg.get("quality_variants", [])
if not isinstance(quality_variants, list):
    quality_variants = []
all_variants = [str(cfg["baseline_variant"])] + [str(v) for v in quality_variants]

print("project_root:", project_root)
print("stage2_root:", stage2_root)
print("all_variants:", all_variants)


## Stage 1: Audit + Shared Split


In [ ]:
from src.group2_stage2.data.audit import audit_stage2_variants
from src.group2_stage2.data.splits import build_shared_quality_pool, materialize_train_val_split

audit = audit_stage2_variants(stage2_root, all_variants)
audit


In [ ]:
pool = build_shared_quality_pool(
    stage2_root=stage2_root,
    all_variants=all_variants,
    quality_image_count=int(cfg["quality_image_count"]),
    val_image_count=int(cfg["val_image_count"]),
    split_seed=int(cfg.get("split_seed", 42)),
    pool_reference_variant=cfg["baseline_variant"],
)
pool


In [ ]:
split_stats = materialize_train_val_split(stage2_root, all_variants)
split_stats


## Stage 2 Checklist (model-dependent)
- [ ] Group1 CLIP bundle loaded
- [ ] Group1 tokenizer loaded
- [ ] `get_features_compiled` ready from Group1 CLIP helper


In [ ]:
from src.group2_stage2.bootstrap_runtime import create_stage2_runtime_objects

runtime = create_stage2_runtime_objects(cfg)
clip_bundle = runtime["clip_bundle"]
get_features_compiled = runtime["get_features_compiled"]
tokenizer = runtime["tokenizer"]

print("runtime ready")
print("clip hidden_size:", clip_bundle.hidden_size)
print("tokenizer loaded:", type(tokenizer).__name__)


In [ ]:
from src.group2_stage2.data.pipeline import prepare_stage2_variant_splits

required = ["tokenizer", "clip_bundle", "get_features_compiled"]
missing = [name for name in required if name not in globals()]

if missing:
    print("Stage 2 prerequisites missing:", missing)
    print("Initialize these first (typically from Group1 runtime):")
    print("- tokenizer")
    print("- clip_bundle")
    print("- get_features_compiled")
else:
    OVERWRITE_STAGE2_PREP = False

    prep_results = prepare_stage2_variant_splits(
        stage2_root=stage2_root,
        image_root=image_root,
        feature_root=feature_root,
        tokenizer=tokenizer,
        clip_bundle=clip_bundle,
        get_features_compiled=get_features_compiled,
        all_variants=all_variants,
        overwrite=OVERWRITE_STAGE2_PREP,
    )
    print("Prepared variant/split jobs:", len(prep_results))
    print(prep_results[0] if prep_results else "<no results>")


## Stage 3: Evaluation Artifacts


In [ ]:
from src.group2_stage2.eval.quality_eval import (
    build_dataset_quality_diagnostics,
    build_qualitative_samples_pack,
    build_pairwise_judge_requests,
)

quality = build_dataset_quality_diagnostics(stage2_root, all_variants)
qual_samples = build_qualitative_samples_pack(stage2_root, all_variants)

# Requires heldout_eval_pack.json already prepared:
# pairwise = build_pairwise_judge_requests(stage2_root, baseline_variant=cfg["baseline_variant"])


## Stage 4: Quality Experiment Tracking
Use these after stage2 manifests/training are available.


In [ ]:
from src.group2_stage2.experiments.experiment_tracking import (
    select_next_variant,
    run_and_store_variant,
    prompt_alignment_audit,
    build_engine_comparison_summary,
    build_baseline_relative_comparison,
)

results_path = stage2_root / "all_results_manual.json"
selection = select_next_variant(
    results_path=results_path,
    expected_variants=all_variants,
    current_variant=None,
    allow_overwrite=False,
)
selection


In [ ]:
# Wire your stage2 experiment runner here (model-dependent):
# run_and_store_variant(
#     results_path=results_path,
#     all_results=selection["all_results"],
#     selected_variant=selection["selected_variant"],
#     run_stage2_experiment_fn=run_stage2_experiment,
# )

prompt_alignment_audit(stage2_root, all_variants, prompt_reference_variant=all_variants[0])


In [ ]:
# Requires all expected variants in all_results_manual.json
# build_engine_comparison_summary(stage2_root, results_path, expected_variants=all_variants)
# build_baseline_relative_comparison(stage2_root, results_path, baseline_variant=cfg["baseline_variant"], expected_variants=all_variants)


## Stage 5: Quantity Ablation


In [ ]:
from src.group2_stage2.experiments.quantity_ablation import (
    derive_quantity_plan,
    build_quantity_variants,
    register_quantity_variants,
    prepare_quantity_variants,
    run_quantity_experiments,
    summarize_quantity_results,
)

engine_summary = stage2_root / "engine_comparison_summary.json"
if not engine_summary.exists():
    print("Stage 5 blocked: missing", engine_summary)
    print("Run Stage 4 engine summary first.")
else:
    plan = derive_quantity_plan(stage2_root)
    plan


In [ ]:
quantity_variants = build_quantity_variants(
    stage2_root=stage2_root,
    quantity_source_variant=plan["quantity_source_variant"],
    quantity_levels=plan["quantity_levels"],
    quantity_split_seed=plan["quantity_split_seed"],
)
register_quantity_variants(stage2_root, quantity_variants)


In [ ]:
# model-dependent preparation + run
# prepare_quantity_variants(
#     stage2_root,
#     quantity_variants,
#     tokenize_fn=tokenize_stage2_variant,
#     extract_features_fn=extract_stage2_features,
#     build_manifest_fn=build_stage2_manifest,
# )
# run_quantity_experiments(stage2_root, quantity_variants, run_stage2_experiment_fn=run_stage2_experiment)
# summarize_quantity_results(stage2_root)


## Stage 6: Heldout Pack + Pairwise Requests


In [ ]:
from src.group2_stage2.eval.evaluation_pack import build_heldout_eval_pack
from src.group2_stage2.eval.quality_eval import build_pairwise_judge_requests

heldout = build_heldout_eval_pack(stage2_root, all_variants, samples_per_task=10, seed=123)
# pairwise = build_pairwise_judge_requests(stage2_root, baseline_variant=cfg["baseline_variant"], seed=2026)
heldout["num_samples"]


## Stage 7: Reporting Figures


In [ ]:
from src.group2_stage2.eval.reporting import build_engine_plots_and_table

engine_summary = stage2_root / "engine_comparison_summary.json"
quality_diag = stage2_root / "dataset_quality_diagnostics.json"
if not engine_summary.exists() or not quality_diag.exists():
    print("Stage 7 blocked: required files missing")
    print("-", engine_summary, "exists=", engine_summary.exists())
    print("-", quality_diag, "exists=", quality_diag.exists())
else:
    build_engine_plots_and_table(stage2_root)


## Stage 2 Reset Orchestration

- Save a clean Stage-2 state snapshot once (after state objects exist).
- Restore that snapshot before reruns to keep experiments comparable.


In [ ]:
from src.group2_stage2.experiments.training_orchestration import save_stage2_snapshot

reset_root = stage2_root / "_fresh_state_snapshot"
required_state = ["projector_state", "llama_state", "opt_state"]
missing_state = [name for name in required_state if name not in globals()]

if missing_state:
    print("Cannot save snapshot. Missing state objects:", missing_state)
else:
    g = globals()
    snapshot_paths = save_stage2_snapshot(
        reset_root=reset_root,
        projector_state=g["projector_state"],
        llama_state=g["llama_state"],
        opt_state=g["opt_state"],
    )
    print("Saved Stage-2 snapshot:", snapshot_paths)


In [ ]:
from src.group2_stage2.experiments.training_orchestration import load_stage2_snapshot

reset_root = stage2_root / "_fresh_state_snapshot"
if not reset_root.exists():
    print("Snapshot folder not found:", reset_root)
    print("Run the save snapshot cell first.")
else:
    loaded = load_stage2_snapshot(reset_root)
    projector_state = loaded["projector_state"]
    llama_state = loaded["llama_state"]
    opt_state = loaded["opt_state"]
    print("Restored Stage-2 states from:", reset_root)
